# Phát Hiện Tin Nhắn Lừa Đảo (Scam Detection)
### Mô hình: Tự xây dựng Transformer từ đầu (Học tập)
- Tự xây dựng từng thành phần: Embeddings, Attention, FFN, Pooler, Head.
- Khởi tạo trọng số ngẫu nhiên (Random Init), không dùng mô hình pretrained.
- Giữ nguyên 100% dữ liệu sạch và lừa đảo, xử lý lệch nhãn bằng Focal Loss.

In [ ]:
# Cai dat thu vien can thiet
!pip install datasets transformers pyarrow scikit-learn matplotlib seaborn -q

In [ ]:
# Import thu vien he thong
import os, gc, glob, math, time, json, shutil, zipfile
import unicodedata
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Import HuggingFace & Sklearn
from datasets import load_dataset, ClassLabel, load_from_disk
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_curve, auc, precision_recall_curve, roc_auc_score

# Kiem tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Thiet bi: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Ket noi Google Drive de doc ghi
from google.colab import drive
drive.mount('/content/drive')

## 1. Cấu hình hệ thống (CONFIG)

In [ ]:
# Cau hinh toan bo he thong
CONFIG = {
    # Duong dan Google Drive va local
    'zip_path': '/content/drive/MyDrive/Học sâu 252AI31001/Data_Preprocessed.zip',
    'extract_dir': '/content/parquet_files',
    'cache_dir': '/content/hf_cache',
    'tokenized_dir': '/content/tokenized',
    'output_dir': '/content/drive/MyDrive/scam_checkpoints',

    # Kien truc mang tu xay dung
    'model_name': 'vinai/phobert-base', # Chi lay tokenizer
    'max_length': 128,
    'num_layers': 6,
    'hidden_size': 384,
    'num_heads': 6,
    'intermediate_size': 1536,
    'max_pos': 258,
    'vocab_size': 64001,
    'pad_token_id': 1,
    'hidden_dropout': 0.3,
    'attention_dropout': 0.3,
    'layer_norm_eps': 1e-5,
    'num_meta': 3,
    'num_classes': 2,

    # Tham so huan luyen
    'batch_size': 128,
    'grad_accum': 2,
    'epochs': 1,
    'lr': 4e-5,
    'weight_decay': 0.1,
    'warmup_ratio': 0.06,
    'grad_clip': 1.0,

    # Focal loss cho du lieu lech
    'focal_gamma': 2.0,
    'focal_alpha': 0.75,

    # Tham so khac
    'num_proc': 2,
    'eval_every_steps': 1000,
    'save_every_steps': 2000,
    'patience': 10,
    'seed': 42,
    'val_ratio': 0.05,
    'test_ratio': 0.05,
}

# Tao thu muc luu tru
for d in [CONFIG['extract_dir'], CONFIG['tokenized_dir'], CONFIG['output_dir'], CONFIG['cache_dir']]:
    os.makedirs(d, exist_ok=True)

CKPT_PATH = os.path.join(CONFIG['output_dir'], 'latest_checkpoint.pt')
BEST_PATH = os.path.join(CONFIG['output_dir'], 'best_model.pt')
print('Cau hinh hoan tat!')

## 2. Nạp dữ liệu và chia tập phân tầng (Stratified Split)
Giữ nguyên 100% dữ liệu gốc (99.72% Clean, 0.28% Scam).

In [ ]:
# Tim va giai nen tap du lieu
zip_path = CONFIG['zip_path']
if not os.path.exists(zip_path):
    norm_path = unicodedata.normalize('NFC', zip_path)
    if os.path.exists(norm_path):
        zip_path = norm_path
    else:
        found = glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True)
        matched = [f for f in found if 'Preprocess' in f]
        if matched:
            zip_path = matched[0]

# Giai nen neu chua ton tai
if not glob.glob(f"{CONFIG['extract_dir']}/**/*.parquet", recursive=True):
    print(f'Dang giai nen tu: {zip_path}')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(CONFIG['extract_dir'])
    print('Giai nen thanh cong!')

# Nap file parquet
parquet_files = sorted(glob.glob(f"{CONFIG['extract_dir']}/**/*.parquet", recursive=True))
print(f'Tim thay {len(parquet_files)} file parquet')

raw_ds = load_dataset('parquet', data_files=parquet_files, split='train', cache_dir=CONFIG['cache_dir'])
raw_ds = raw_ds.cast_column('label', ClassLabel(num_classes=2, names=['clean', 'scam']))

# Kiem tra so luong
total_n = len(raw_ds)
labels_arr = np.array(raw_ds['label'])
n_0 = int((labels_arr == 0).sum())
n_1 = int((labels_arr == 1).sum())
print(f'Tong: {total_n:,} | Clean: {n_0:,} ({n_0 / total_n * 100:.2f}%) | Scam: {n_1:,} ({n_1 / total_n * 100:.2f}%)')
del labels_arr; gc.collect()

# Chia tap train/val/test theo 90/5/5
holdout = CONFIG['val_ratio'] + CONFIG['test_ratio']
sp1 = raw_ds.train_test_split(test_size=holdout, stratify_by_column='label', seed=CONFIG['seed'])
train_ds = sp1['train']
sp2 = sp1['test'].train_test_split(test_size=0.5, stratify_by_column='label', seed=CONFIG['seed'])
val_ds = sp2['train']
test_ds = sp2['test']
print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
del raw_ds, sp1, sp2; gc.collect()

## 3. Mã hóa văn bản và chuẩn hóa Metadata

In [ ]:
# Load tokenizer tu vinai/phobert-base
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

# Chuan hoa log n_subscribers tren train
max_subs = float(max(train_ds['n_subscribers']) or 1.0)
meta_info = {'max_subs': max_subs, 'max_length': CONFIG['max_length']}
with open(os.path.join(CONFIG['output_dir'], 'meta_config.json'), 'w', encoding='utf-8') as f:
    json.dump(meta_info, f, indent=2)
print(f'Max subs tap train: {max_subs:,}')


# Ham tokenize va trich metadata
def tokenize_fn(batch):
    enc = tokenizer(
        batch['cleaned_text'],
        max_length=CONFIG['max_length'],
        padding='max_length',
        truncation=True
    )
    log_max = math.log1p(max_subs) if max_subs > 0 else 1.0
    meta = [
        [float(u), float(v), math.log1p(float(s)) / log_max]
        for u, v, s in zip(batch['has_url'], batch['verified'], batch['n_subscribers'])
    ]
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'metadata': meta,
        'label': batch['label']
    }


# Luu tokenized dataset
for name, dset in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    out_path = os.path.join(CONFIG['tokenized_dir'], name)
    if not os.path.exists(out_path):
        print(f'Dang xu ly {name}...')
        tok_data = dset.map(
            tokenize_fn,
            batched=True,
            batch_size=50000,
            num_proc=CONFIG['num_proc'],
            remove_columns=['cleaned_text', 'has_url', 'verified', 'n_subscribers'],
            desc=f'Tokenize {name}'
        )
        tok_data.save_to_disk(out_path)
        del tok_data; gc.collect()

del train_ds, val_ds, test_ds; gc.collect()


print('Luu tokenized dataset thanh cong!')

## 4. Tự xây dựng kiến trúc mạng Transformer từ đầu
Không mượn cấu trúc hay trọng số từ bất kỳ thư viện pretrained nào.

In [ ]:
# 1. Lop Embeddings
class Embeddings(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.word_emb = nn.Embedding(cfg['vocab_size'], cfg['hidden_size'], padding_idx=cfg['pad_token_id'])
        self.pos_emb = nn.Embedding(cfg['max_pos'], cfg['hidden_size'])
        self.type_emb = nn.Embedding(2, cfg['hidden_size'])
        self.layer_norm = nn.LayerNorm(cfg['hidden_size'], eps=cfg['layer_norm_eps'])
        self.dropout = nn.Dropout(cfg['hidden_dropout'])
        self.register_buffer('pos_ids', torch.arange(cfg['max_pos']).expand((1, -1)))
        
    def forward(self, input_ids):
        seq_len = input_ids.size(1)
        w = self.word_emb(input_ids)
        p = self.pos_emb(self.pos_ids[:, :seq_len])
        t = self.type_emb(torch.zeros_like(input_ids))
        return self.dropout(self.layer_norm(w + p + t))

# 2. Multi-Head Self-Attention
class SelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.num_heads = cfg['num_heads']
        self.head_dim = cfg['hidden_size'] // cfg['num_heads']
        self.q = nn.Linear(cfg['hidden_size'], cfg['hidden_size'])
        self.k = nn.Linear(cfg['hidden_size'], cfg['hidden_size'])
        self.v = nn.Linear(cfg['hidden_size'], cfg['hidden_size'])
        self.dropout = nn.Dropout(cfg['attention_dropout'])
        
    def forward(self, x, mask):
        B, L, H = x.size()
        q = self.q(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores + mask
        attn = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous()
        return out.view(B, L, H)

# 3. Attention Block voi residual va LN
class AttentionBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn = SelfAttention(cfg)
        self.proj = nn.Linear(cfg['hidden_size'], cfg['hidden_size'])
        self.layer_norm = nn.LayerNorm(cfg['hidden_size'], eps=cfg['layer_norm_eps'])
        self.dropout = nn.Dropout(cfg['hidden_dropout'])
        
    def forward(self, x, mask):
        a = self.attn(x, mask)
        return self.layer_norm(x + self.dropout(self.proj(a)))

# 4. Feed-Forward Network
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg['hidden_size'], cfg['intermediate_size'])
        self.fc2 = nn.Linear(cfg['intermediate_size'], cfg['hidden_size'])
        self.layer_norm = nn.LayerNorm(cfg['hidden_size'], eps=cfg['layer_norm_eps'])
        self.dropout = nn.Dropout(cfg['hidden_dropout'])
        
    def forward(self, x):
        a = self.dropout(self.fc2(F.gelu(self.fc1(x))))
        return self.layer_norm(x + a)

# 5. Lop Transformer
class TransformerLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn = AttentionBlock(cfg)
        self.ff = FeedForward(cfg)
        
    def forward(self, x, mask):
        return self.ff(self.attn(x, mask))

# 6. Mạng hoàn chỉnh ScamTransformer
class ScamTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.embed = Embeddings(cfg)
        self.encoder = nn.ModuleList([TransformerLayer(cfg) for _ in range(cfg['num_layers'])])
        self.pooler = nn.Linear(cfg['hidden_size'], cfg['hidden_size'])
        
        self.meta_proj = nn.Sequential(
            nn.Linear(cfg['num_meta'], 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Linear(cfg['hidden_size'] + 32, 128), nn.ReLU(),
            nn.Dropout(0.1), nn.Linear(128, cfg['num_classes'])
        )
        self.apply(self._init_weights)
        
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: m.bias.data.zero_()
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.LayerNorm):
            m.weight.data.fill_(1.0)
            m.bias.data.zero_()
            
    def _extend_mask(self, mask):
        return (1.0 - mask[:, None, None, :].float()) * -10000.0
        
    def forward(self, input_ids, mask, meta):
        x = self.embed(input_ids)
        emask = self._extend_mask(mask)
        for layer in self.encoder:
            x = layer(x, emask)
        cls_out = torch.tanh(self.pooler(x[:, 0, :]))
        meta_out = self.meta_proj(meta)
        combined = torch.cat([cls_out, meta_out], dim=-1)
        return self.head(combined)

## 5. Hàm mất mát Focal Loss

In [ ]:
# Focal Loss xu ly lech nhan
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.75):
        super().__init__()
        self.gamma = gamma
        self.register_buffer('alpha_w', torch.tensor([1.0 - alpha, alpha], dtype=torch.float32))
        
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        at = self.alpha_w[targets]
        fl = at * ((1.0 - pt) ** self.gamma) * ce
        return fl.mean()

## 6. Dataset và DataLoader

In [ ]:
# PyTorch Dataset tu Arrow
class ArrowDataset(Dataset):
    def __init__(self, hf_ds):
        hf_ds.set_format('torch')
        self.ds = hf_ds
        
    def __len__(self):
        return len(self.ds)
        
    def __getitem__(self, i):
        row = self.ds[i]
        return (
            row['input_ids'].long(),
            row['attention_mask'].long(),
            row['metadata'].float(),
            row['label'].long()
        )

# Tao cac bo DataLoader
train_set = load_from_disk(os.path.join(CONFIG['tokenized_dir'], 'train'))
val_set = load_from_disk(os.path.join(CONFIG['tokenized_dir'], 'val'))
test_set = load_from_disk(os.path.join(CONFIG['tokenized_dir'], 'test'))

train_loader = DataLoader(
    ArrowDataset(train_set),
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_proc'],
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)
val_loader = DataLoader(
    ArrowDataset(val_set),
    batch_size=CONFIG['batch_size'] * 2,
    shuffle=False,
    num_workers=CONFIG['num_proc'],
    pin_memory=True
)
test_loader = DataLoader(
    ArrowDataset(test_set),
    batch_size=CONFIG['batch_size'] * 2,
    shuffle=False,
    num_workers=CONFIG['num_proc'],
    pin_memory=True
)
print(f'Batch counts: Train={len(train_loader):,}, Val={len(val_loader):,}, Test={len(test_loader):,}')

## 7. Khởi tạo Optimizer, Scheduler và Checkpoint

In [ ]:
# Khoi tao mo hinh
model = ScamTransformer(CONFIG).to(device)
criterion = FocalLoss(CONFIG['focal_gamma'], CONFIG['focal_alpha'])
print(f'Tong tham so tu xay: {sum(p.numel() for p in model.parameters()):,}')

# Weight decay separation
no_decay = ['bias', 'LayerNorm.weight', 'layer_norm.weight']
opt_params = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': CONFIG['weight_decay']},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0}
]
optimizer = torch.optim.AdamW(opt_params, lr=CONFIG['lr'])

total_steps = (len(train_loader) // CONFIG['grad_accum']) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = torch.amp.GradScaler('cuda')

# Luu va tai checkpoint
def save_ckpt(path, state):
    tmp = path + '.tmp'
    torch.save(state, tmp)
    shutil.move(tmp, path)
    
def load_ckpt(path):
    return torch.load(path, map_location='cpu', weights_only=False)

## 8. Hàm đánh giá và tìm ngưỡng tối ưu

In [ ]:
# Ham danh gia va tim nguong
@torch.no_grad()
def evaluate(loader, split_name='val'):
    model.eval()
    all_probs, all_labels, losses = [], [], []
    
    for ids, mask, meta, lbl in loader:
        ids, mask = ids.to(device), mask.to(device)
        meta, lbl = meta.to(device), lbl.to(device)
        
        with torch.amp.autocast('cuda'):
            out = model(ids, mask, meta)
            loss = criterion(out, lbl)
            
        losses.append(loss.item())
        probs = torch.softmax(out, dim=1)[:, 1]
        all_probs.extend(probs.cpu().tolist())
        all_labels.extend(lbl.cpu().tolist())
        
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    # Quet nguong de tim F1 tot nhat
    best_f1, best_th = 0.0, 0.50
    for th in np.arange(0.01, 0.51, 0.01):
        preds = (all_probs >= th).astype(int)
        f1 = f1_score(all_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_th = th
            
    avg_loss = float(np.mean(losses))
    print(f'[{split_name}] Loss: {avg_loss:.4f} | F1: {best_f1:.4f} (Nguong: {best_th:.2f})')
    return avg_loss, best_f1, best_th

## 9. Vòng lặp huấn luyện (Training Loop)

In [ ]:
# Vong lap huan luyen
history = {'val_loss': [], 'val_f1': [], 'steps': []}
step, best_val_f1, no_improve = 0, 0.0, 0

if os.path.exists(CKPT_PATH):
    c = load_ckpt(CKPT_PATH)
    model.load_state_dict(c['model'])
    optimizer.load_state_dict(c['opt'])
    scheduler.load_state_dict(c['sch'])
    scaler.load_state_dict(c['scaler'])
    step = c['step']
    best_val_f1 = c['best_f1']
    print(f'Resume tu step {step:,} (Best F1: {best_val_f1:.4f})')

model.train()
for ep in range(CONFIG['epochs']):
    for i, (ids, mask, meta, lbl) in enumerate(train_loader):
        ids, mask = ids.to(device), mask.to(device)
        meta, lbl = meta.to(device), lbl.to(device)
        
        with torch.amp.autocast('cuda'):
            logits = model(ids, mask, meta)
            loss = criterion(logits, lbl) / CONFIG['grad_accum']
            
        scaler.scale(loss).backward()
        
        if (i + 1) % CONFIG['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step += 1
            
            if step % 100 == 0:
                print(f'Step {step:,} | Loss: {loss.item()*CONFIG["grad_accum"]:.4f}')
                
            if step % CONFIG['eval_every_steps'] == 0:
                vl, vf, vth = evaluate(val_loader, 'val')
                history['val_loss'].append(vl)
                history['val_f1'].append(vf)
                history['steps'].append(step)
                
                if vf > best_val_f1:
                    best_val_f1 = vf
                    no_improve = 0
                    save_ckpt(BEST_PATH, {'model': model.state_dict(), 'best_th': vth, 'best_f1': vf})
                    print(f'Luu model moi (F1: {best_val_f1:.4f})')
                else:
                    no_improve += 1
                    
            if step % CONFIG['save_every_steps'] == 0:
                save_ckpt(CKPT_PATH, {
                    'model': model.state_dict(), 'opt': optimizer.state_dict(),
                    'sch': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                    'step': step, 'best_f1': best_val_f1
                })
                
            if no_improve >= CONFIG['patience']:
                print('Dung som!')
                break
                
    if no_improve >= CONFIG['patience']:
        break

with open(os.path.join(CONFIG['output_dir'], 'history.json'), 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)

## 10. Đánh giá chi tiết trên tập Test

In [ ]:
# Danh gia tren test set
if os.path.exists(BEST_PATH):
    b_data = load_ckpt(BEST_PATH)
    model.load_state_dict(b_data['model'])
    test_th = b_data.get('best_th', 0.50)
else:
    test_th = 0.50

model.eval()
test_probs, test_labels = [], []
with torch.no_grad():
    for ids, mask, meta, lbl in test_loader:
        ids, mask = ids.to(device), mask.to(device)
        meta = meta.to(device)
        with torch.amp.autocast('cuda'):
            logits = model(ids, mask, meta)
        p = torch.softmax(logits, dim=1)[:, 1]
        test_probs.extend(p.cpu().tolist())
        test_labels.extend(lbl.tolist())

test_probs = np.array(test_probs)
test_labels = np.array(test_labels)
test_preds = (test_probs >= test_th).astype(int)

print(classification_report(test_labels, test_preds, target_names=['Clean', 'Scam'], digits=4))
print(f'ROC-AUC: {roc_auc_score(test_labels, test_probs):.4f}')

# Ve confusion matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Clean', 'Scam'], yticklabels=['Clean', 'Scam'])
plt.title('Confusion Matrix - Test')
plt.savefig(os.path.join(CONFIG['output_dir'], 'cm_test.png'), dpi=300)
plt.show()

# Ve ROC curve
fpr, tpr, _ = roc_curve(test_labels, test_probs)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc(fpr, tpr):.4f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve')
plt.legend()
plt.savefig(os.path.join(CONFIG['output_dir'], 'roc_test.png'), dpi=300)
plt.show()

## 11. Đồ thị tiến trình huấn luyện

In [ ]:
# Ve duong cong hoc
hist_p = os.path.join(CONFIG['output_dir'], 'history.json')
if os.path.exists(hist_p):
    with open(hist_p, 'r', encoding='utf-8') as f:
        h = json.load(f)
    if h.get('steps'):
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(h['steps'], h['val_loss'], 'r-')
        ax[0].set_title('Val Loss')
        ax[0].grid(True)
        ax[1].plot(h['steps'], h['val_f1'], 'b-')
        ax[1].set_title('Val F1')
        ax[1].grid(True)
        plt.savefig(os.path.join(CONFIG['output_dir'], 'curves.png'), dpi=300)
        plt.show()

## 12. Thử nghiệm dự đoán thực tế (Demo Inference)
Không dùng từ khóa hay hardcode điểm số, hoàn toàn suy luận bằng mạng.

In [ ]:
# Ham du doan khong hack diem
def predict(texts, has_url=None, verified=None, n_subs=None, threshold=0.5):
    if isinstance(texts, str): texts = [texts]
    n = len(texts)
    has_url = has_url if has_url is not None else [0.0] * n
    verified = verified if verified is not None else [0.0] * n
    n_subs = n_subs if n_subs is not None else [0.0] * n
    
    cfg_f = os.path.join(CONFIG['output_dir'], 'meta_config.json')
    m_s = 1.0
    if os.path.exists(cfg_f):
        with open(cfg_f, 'r', encoding='utf-8') as f:
            m_s = json.load(f).get('max_subs', 1.0)
    log_m_s = math.log1p(m_s) if m_s > 0 else 1.0
    
    model.eval()
    res = []
    for i in range(0, n, 32):
        b_txt = texts[i:i+32]
        tok = tokenizer(b_txt, max_length=CONFIG['max_length'], padding='max_length', truncation=True, return_tensors='pt')
        b_meta = [
            [float(has_url[j]), float(verified[j]), math.log1p(float(n_subs[j])) / log_m_s]
            for j in range(i, min(i+32, n))
        ]
        b_ids = tok['input_ids'].to(device)
        b_mask = tok['attention_mask'].to(device)
        b_meta_t = torch.tensor(b_meta, dtype=torch.float32).to(device)
        
        with torch.no_grad(), torch.amp.autocast('cuda'):
            logits = model(b_ids, b_mask, b_meta_t)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().tolist()
            
        for p in probs:
            label = 'SCAM' if p >= threshold else 'CLEAN'
            res.append((label, p))
    return res

# Mau tin nhan thu nghiem
samples = [
    'Chuc mung ban da trung thuong 100 trieu! Nhan giai ngay.',
    'Hom nay troi dep, chieu nay di da bong khong ae?',
    'Dau tu tien ao sinh loi 200% thang, cam ket hoan von.',
    'Tai khoan cua ban bi khoa, bam vao link de mo khoa.',
    'Toi nay may gio di an com the?',
    'Tuyen CTV lam viec online tai nha 500k mot ngay.',
    'ShopeeFood giao don com ga, moi quy khach nhan hang.',
    'Canh bao he thong dinh danh VNeID bi khoa, xac thuc ngay.',
    'Em kiem tra giup anh bao cao doanh thu nhe.',
    'Co hoi nhan qua tri an mien phi tu ngan hang.'
]
s_urls = [1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0]
s_vers = [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0]
s_subs = [0.0, 200.0, 10.0, 0.0, 500.0, 0.0, 50000.0, 0.0, 2000.0, 0.0]

preds = predict(samples, s_urls, s_vers, s_subs, threshold=0.30)
for txt, (lbl, conf) in zip(samples, preds):
    print(f'[{lbl:<5} | {conf:.4f}] {txt}')

## 13. Xuất kết quả và sao lưu

In [ ]:
# Luu bao cao va nen checkpoints
perf_rep = {'status': 'success', 'model': 'CustomTransformer'}
with open(os.path.join(CONFIG['output_dir'], 'perf.json'), 'w') as f:
    json.dump(perf_rep, f)
    
zip_out = os.path.join(CONFIG['output_dir'], 'checkpoints_backup')
shutil.make_archive(zip_out, 'zip', CONFIG['output_dir'])
print('Hoan tat qua trinh!')